# Predict a horizon class from a user prompt

Takes a user prompt (a plain `str`), formats it with the model's chat template,
caches the residual-stream activation at the output of transformer layer 21
(`layer_out/21`) for the final prompt token, and pushes that vector through the
saved projection pipeline:

    activation -> PLS1-3 + residual PC1-3 -> surface coordinates (t, u) -> horizon class

Nothing is fitted here and nothing is uploaded. Extraction lives in
`vendor/activations/`; the projection models are the artifacts under `models/`
written by `projection_direciton.ipynb`.

The residual PCA is the one artifact that notebook never wrote -- it was fit on
the fly inside the Activation Atlas app. `vendor/activations/fit_residual_pca.py`
rebuilds it from the cached activations in `.acts/`; rerun that script if
`models/ctype_only_residual_pca_layer_out-21.joblib` is missing.

## Setup

In [ ]:
# Reload edited modules automatically: the helpers under `vendor/` change often,
# and without this an already-imported module keeps its stale copy for the life
# of the kernel.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import torch

# The saved models unpickle classes from `temporal_manifolds`; a minimal copy of
# that code lives in `vendor/` at the repo root.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "vendor").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "vendor") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "vendor"))

from activations.extraction import (
    PROMPT_TOKEN_POSITION,
    TARGET_LAYER_COMPONENT,
    extract_last_position_activation,
    load_model_tokenizer,
)
from activations.residual_pca import load_residual_pca, project_activations
from temporal_manifolds.viz.activation_pls import load_pls_model
from temporal_manifolds.viz.extruded_surface import load_surface_model
from utils.horizon_classes import HORIZON_CLASS_LABELS
from utils.ordinal_regression import load_ordinal_model

print(f"repo root: {REPO_ROOT}")
print(f"target: {TARGET_LAYER_COMPONENT} @ position {PROMPT_TOKEN_POSITION}")

## Configuration

`HF_TOKEN` is read from the environment if left as `None`; set it inline only if
you are not using an environment variable. It is needed for gated repos and to
raise the download rate limit, and is passed straight to `transformers`.

`MODEL_NAME` must be the model the PLS artifact was fit on -- the projection is
meaningless in another model's activation space. The artifact's own metadata is
checked against the loaded model below.

`OFFSET_DEGREE = 2` and `DEGREE = 3` match the artifacts written by
`projection_direciton.ipynb`.

In [ ]:
import os

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
HF_TOKEN = os.environ.get("HF_TOKEN")  # or paste the token string here

DEVICE = None       # None -> cuda, else mps, else cpu
DTYPE = "bfloat16"  # None keeps the checkpoint dtype
SYSTEM_PROMPT = ""  # empty means no system turn is added, as in the cached runs
THINKING = False    # chat-template `enable_thinking`

OFFSET_DEGREE = 2   # extrusion profile degree, picks the surface artifact
DEGREE = 3          # classifier polynomial degree, picks the model artifact

PLS_PATH = REPO_ROOT / "models" / "ctype_only_activation_pls_layer_out-21.joblib"
RESIDUAL_PCA_PATH = REPO_ROOT / "models" / "ctype_only_residual_pca_layer_out-21.joblib"
SURFACE_PATH = (
    REPO_ROOT / "models"
    / f"ctype_only_activation_surface_PLS1-PLS2-by-t_extruded-PLS3_degree-{OFFSET_DEGREE}.joblib"
)
CLASSIFIER_PATH = (
    REPO_ROOT / "models"
    / f"ctype_only_horizon_class_ordinal_binary-decomposition_degree-{DEGREE}.joblib"
)

PARAMETER_SAMPLES = 4000   # density of the nearest-t search when projecting

print(f"model: {MODEL_NAME}")
print(f"HF token: {'set' if HF_TOKEN else 'not set'}")

## Load the projection models

In [ ]:
pls, pls_metadata = load_pls_model(PLS_PATH)
residual_pca, residual_metadata = load_residual_pca(RESIDUAL_PCA_PATH)
surface, surface_metadata = load_surface_model(SURFACE_PATH)
classifier, classifier_features, classifier_metadata = load_ordinal_model(CLASSIFIER_PATH)

print("pls       :", PLS_PATH.name)
print("            ", pls_metadata["layer_component"],
      "| position", pls_metadata["cached_position"],
      "| components", pls_metadata["component_count"],
      "| activation width", pls_metadata["feature_count"])
print("residual  :", RESIDUAL_PCA_PATH.name)
print("            components", residual_pca.components_.shape[0],
      "| agreement with the stored residual PCs",
      np.round(residual_metadata["stored_pc_correlations"], 6))
print("surface   :", SURFACE_PATH.name)
print("            offset degree", surface.offset_degree,
      "| t range", np.round(surface.training_parameter_bounds, 3))
print("classifier:", CLASSIFIER_PATH.name)
print("            features", classifier_features)
print("            training accuracy",
      round(classifier_metadata["training_metrics"]["accuracy"], 4))

## Load the model

Downloads the checkpoint from the Hugging Face Hub on first run and reuses the
local cache afterwards.

In [ ]:
model, tokenizer = load_model_tokenizer(
    MODEL_NAME,
    hf_token=HF_TOKEN,
    device=DEVICE,
    dtype=DTYPE,
    system_prompt=SYSTEM_PROMPT,
)

hidden_size = int(model.config.hidden_size)
if hidden_size != pls_metadata["feature_count"]:
    raise ValueError(
        f"{MODEL_NAME} has hidden size {hidden_size}, but the PLS artifact was fit on "
        f"{pls_metadata['feature_count']}-dimensional activations."
    )

print(f"layers: {model.config.num_hidden_layers}, hidden size: {hidden_size}")
print(f"device: {next(model.parameters()).device}, dtype: {next(model.parameters()).dtype}")

## Prompt

In [ ]:
PROMPT = "Could you help me plan how to write a wedding speech? I have 2 weeks available."

formatted = tokenizer._apply_chat_template(PROMPT, thinking=THINKING)[0]
print(formatted)

## Extract and project

One forward pass, then the coordinates the classifier was trained on: PLS
scores, the principal components of what those scores fail to reconstruct, and
the surface parameters `(t, u)` of the resulting point.

In [ ]:
activation = extract_last_position_activation(
    model,
    tokenizer,
    PROMPT,
    thinking=THINKING,
)
print(f"{TARGET_LAYER_COMPONENT} @ position {PROMPT_TOKEN_POSITION}:",
      f"shape {tuple(activation.shape)}, norm {activation.norm().item():.4f}")

scores, residual_scores, residual_rms = project_activations(
    pls, residual_pca, activation.numpy()
)
t, u, surface_distance = surface.project(scores, parameter_samples=PARAMETER_SAMPLES)

features = {
    "PLS1": scores[0, 0], "PLS2": scores[0, 1], "PLS3": scores[0, 2],
    "t": t[0], "u": u[0],
    "reconstruction_residual_PC1": residual_scores[0, 0],
    "reconstruction_residual_PC2": residual_scores[0, 1],
    "reconstruction_residual_PC3": residual_scores[0, 2],
    "reconstruction_residual_rms": residual_rms[0],
    "surface_distance": surface_distance[0],
}
for name, value in features.items():
    print(f"  {name:<28} {value: .4f}")

lower, upper = surface.training_parameter_bounds
if not lower <= t[0] <= upper:
    print(f"\nwarning: t is outside the fitted range [{lower:.3f}, {upper:.3f}]")

## Predicted class

One caveat on the number: every point the classifier was trained on is an
average over dozens of prompt variants sharing a task and a horizon, while this
is a single prompt, so it sits further from the fitted surface than a training
point typically does.

In [ ]:
feature_vector = np.array([[features[name] for name in classifier_features]], dtype=float)
predicted = int(classifier.predict(feature_vector)[0])
predicted_label = HORIZON_CLASS_LABELS[predicted]

if hasattr(classifier, "predict_proba"):
    probabilities = np.asarray(classifier.predict_proba(feature_vector))[0]
    ranked = np.argsort(probabilities)[::-1][:3]
    print("most likely classes:")
    for index in ranked:
        print(f"  {HORIZON_CLASS_LABELS[index]:<20} {probabilities[index]:.3f}")
    print()

print(f"prompt          : {PROMPT}")
print(f"predicted class : {predicted} - {predicted_label}")